# LeetCode 560 - Subarray Sum Equals K

## Problem

Given an integer array `nums` and an integer `k`, return the **total number of continuous subarrays** whose sum equals `k`.

Unlike Sliding Window, the array may contain:

- Positive numbers
- Negative numbers
- Zero

---

# Brute Force

Generate every possible subarray.

For each subarray:

1. Compute its sum.
2. If the sum equals `k`, increment the answer.

### Time Complexity

```
O(n²)
```

Using a running sum inside the inner loop.

---

# Why Sliding Window Fails

Sliding Window only works when all numbers are **positive**.

Example:

```
[1, -1, 2]
```

When negative numbers exist,

- Expanding the window can decrease the sum.
- Shrinking the window can increase the sum.

Therefore,

```
Sum is NOT monotonic.
```

Sliding Window cannot decide whether to expand or shrink.

---

# Key Observation

Instead of storing subarrays,

store the **running prefix sums**.

---

## Prefix Sum

Prefix Sum = Sum from index `0` to the current index.

Example

```
nums = [1,2,3,4]
```

| Index | Prefix Sum |
|-------:|-----------:|
| Before Array | 0 |
| 0 | 1 |
| 1 | 3 |
| 2 | 6 |
| 3 | 10 |

---

# Main Idea

Suppose the current prefix sum is

```
Current Prefix = 10
```

We want a subarray whose sum is

```
k = 7
```

Then

```
Current Prefix - Previous Prefix = k
```

Rearranging,

```
Previous Prefix = Current Prefix - k
```

So at every index we ask:

> **Have I seen a previous prefix sum equal to `Current Prefix - k`?**

If yes,

then a valid subarray exists.

---

# Example

```
nums = [1,2,3,4]
k = 7
```

Current Prefix

```
10
```

Need

```
10 - 7 = 3
```

A previous prefix sum of `3` exists.

Therefore,

```
10 - 3 = 7
```

Subarray:

```
[3,4]
```

---

# Why HashMap?

A **set** only tells us:

```
Have I seen this prefix sum?
```

But the problem asks:

```
How many subarrays exist?
```

The same prefix sum can appear multiple times.

Example

```
Prefix Sum 0 appears 3 times.
```

Each occurrence creates a different valid subarray.

Therefore we store

```
Prefix Sum → Frequency
```

instead of

```
Prefix Sum → Exists
```

---

# Why `count += prefix_map[need]`

Suppose

```
Current Prefix = 20

Need = 13
```

HashMap

```
13 → 4
```

Meaning

```
Prefix Sum 13 has appeared 4 times.
```

Each previous occurrence creates a different valid subarray ending at the current index.

Therefore

```
count += 4
```

NOT

```
count += 1
```

---

# Why Initialize

```
prefix_map = {0:1}
```

Before processing any element,

the running prefix sum is

```
0
```

This represents the empty prefix before the array starts.

Without it,

subarrays beginning from index `0` would never be counted.

Example

```
nums = [3]
k = 3
```

Current Prefix

```
3
```

Need

```
3 - 3 = 0
```

Since

```
0 → 1
```

already exists,

the subarray

```
[3]
```

is counted correctly.

---

# Algorithm

1. Initialize

- Running Prefix Sum
- Count
- HashMap with `{0:1}`

2. Traverse the array.

3. Update the running prefix sum.

4. Compute

```
need = prefix - k
```

5. If `need` exists in the hashmap,

increase the answer by

```
prefix_map[need]
```

6. Store the current prefix sum by increasing its frequency.

7. Return the final count.

---

# Dry Run

```
nums = [1,1,1]
k = 2
```

### Initial

```
prefix = 0

HashMap

{
0:1
}
```

---

### Read 1

```
prefix = 1

need = -1
```

Not found.

Store

```
{
0:1,
1:1
}
```

---

### Read 1

```
prefix = 2

need = 0
```

Found

```
0 → 1
```

```
count = 1
```

Store

```
{
0:1,
1:1,
2:1
}
```

---

### Read 1

```
prefix = 3

need = 1
```

Found

```
1 → 1
```

```
count = 2
```

Store

```
{
0:1,
1:1,
2:1,
3:1
}
```

Answer

```
2
```

---

# Time Complexity

```
O(n)
```

Each element is processed exactly once.

---

# Space Complexity

```
O(n)
```

The hashmap may store up to `n` different prefix sums.

---

# Pattern Learned

**Prefix Sum + HashMap**

---

# Common Mistakes

### 1. Using Sliding Window

❌ Incorrect because negative numbers are allowed.

Sliding Window requires positive numbers.

---

### 2. Using a Set

A set only tells us

```
Exists or Not
```

We need

```
Frequency
```

because multiple previous prefixes can generate multiple valid subarrays.

---

### 3. Forgetting `{0:1}`

Without

```
{0:1}
```

subarrays starting from index `0` will never be counted.

---

### 4. Doing

```
count += 1
```

Wrong.

Use

```
count += prefix_map[need]
```

because the same prefix sum may have appeared multiple times.

---

# Pattern Recognition

| Condition | Pattern |
|-----------|---------|
| Positive numbers only | Sliding Window |
| Positive + Negative numbers | Prefix Sum + HashMap |
| Count Subarrays | Prefix Sum + HashMap |
| Prefix Sum appears multiple times | Store Frequency in HashMap |

---

# Interview Takeaway

Whenever you see:

- Count subarrays
- Sum equals `k`
- Negative numbers are allowed

Think:

> **Prefix Sum + HashMap**

Remember:

You are **not searching for subarrays**.

You are searching for **previous prefix sums** such that:

```
Current Prefix - Previous Prefix = k
```

Every matching previous prefix represents one valid starting point for a subarray ending at the current index.

In [ ]:
class Solution(object):
    def subarraySum(self, nums, k):
        """
        :type nums: List[int]
        :type k: int
        :rtype: int
        """
        prefix=0
        count=0
        prefix_map={0:1}
        for i in nums:
            prefix+=i
            need=prefix-k

            if need in prefix_map:
                count+=prefix_map[need]
            if prefix in prefix_map:
                prefix_map[prefix]+=1
            else:
                prefix_map[prefix]=1
        return count